# <font color="#418FDE" size="6.5" uppercase>**Data Contracts**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Prepare NumPy, pandas, and sparse inputs that satisfy scikit-learn shape requirements. 
- Identify dtype and target-shape issues before fitting estimators. 
- Use dataset containers, feature names, cloning, tags, and diagrams for inspection. 


## **1. Input Formats**

### **1.1. NumPy Array Shapes**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_01_01.jpg?v=1783903115" width="250">



>* Rows are samples; columns are features
>* Single features still need two-dimensional shape

>* Reshape one-dimensional features into columns
>* Correct shape defines data meaning

>* Targets usually have one value per sample
>* Match feature rows to target entries



In [ ]:
#@title Python Code - NumPy Array Shapes

# NumPy shapes define sample feature meaning.
# Scikit estimators expect rectangular feature matrices.
# Targets usually keep one value per sample.

import numpy as np

# Create one feature measured for five houses.
house_sizes_sqft = np.array([850, 920, 1100, 1350, 1600])

# Create one target value for each house.
prices_thousands = np.array([210, 230, 275, 330, 390])

# Reshape features into rows and columns.
X_column = house_sizes_sqft.reshape(-1, 1)

# Reshape differently to show the common mistake.
X_row = house_sizes_sqft.reshape(1, -1)

# Check that samples match target entries.
valid_pair = X_column.shape[0] == prices_thousands.shape[0]

# Print only compact shape information.
print("Original one-dimensional feature shape:", house_sizes_sqft.shape)
print("Correct feature matrix shape:", X_column.shape)
print("Incorrect single-sample row shape:", X_row.shape)
print("Target vector shape:", prices_thousands.shape)
print("Do feature rows match target values?", valid_pair)
print("Meaning: five samples, one feature each.")



### **1.2. DataFrame Inputs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_01_02.jpg?v=1783903125" width="250">



>* Rows are observations; columns are features
>* Use two-dimensional tables, even for one feature

>* Column names make features easier to audit
>* Keep rows aligned and exclude identifiers

>* Preprocess mixed column types before estimation
>* Validate rows, features, and estimator-ready values



In [ ]:
#@title Python Code - DataFrame Inputs

# DataFrames make estimator inputs easier to inspect.
# Rows are samples, and columns are features.
# Targets must align with the same rows.

import numpy as np
import pandas as pd

# Build a tiny rectangular feature table.
features = pd.DataFrame(
    {
        "area_sqft": [850, 1200, 950, 1600],
        "distance_km": [2.1, 5.4, 3.2, 8.0],
        "bedrooms": [1, 3, 2, 4],
    }
)

# Store the target separately but aligned.
target = pd.Series(
    [240000, 410000, 300000, 520000],
    name="price_usd",
)

# Check the two-dimensional feature contract.
feature_shape_ok = len(features.shape) == 2
row_alignment_ok = features.shape[0] == target.shape[0]

# Check that feature columns are numeric.
numeric_columns = features.select_dtypes(include=[np.number])
all_numeric_ok = numeric_columns.shape[1] == features.shape[1]

# Show compact inspection results only.
print("Feature shape:", features.shape)
print("Target shape:", target.shape)
print("Feature names:", list(features.columns))
print("Two-dimensional features:", feature_shape_ok)
print("Rows aligned with target:", row_alignment_ok)
print("All feature columns numeric:", all_numeric_ok)

# Demonstrate the common one-column DataFrame pattern.
one_feature = features[["area_sqft"]]
print("One-feature DataFrame shape:", one_feature.shape)



### **1.3. Sparse Matrix Inputs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_01_03.jpg?v=1783903127" width="250">



>* Sparse inputs save memory for mostly-zero features
>* Rows are samples; columns are features

>* Use two-dimensional sparse feature matrices
>* Keep rows as samples, columns as features

>* Choose CSR or CSC for operations
>* Avoid unsupported estimators and densifying preprocessing



In [ ]:
#@title Python Code - Sparse Matrix Inputs

# Sparse matrices keep mostly zero features compact.
# Scikit learn expects two dimensional feature inputs.
# This example checks shape and memory safely.

import numpy as np
from scipy import sparse

# Create tiny sample by feature data.
rows = np.array([0, 0, 1, 2, 3])
cols = np.array([1, 4, 2, 0, 3])
values = np.array([2.0, 1.0, 3.0, 5.0, 4.0])

# Build compressed sparse row feature matrix.
X_sparse = sparse.csr_matrix((values, (rows, cols)), shape=(4, 6))
y_target = np.array([0, 1, 0, 1])

# Validate the estimator style data contract.
is_two_dimensional = X_sparse.ndim == 2
matching_samples = X_sparse.shape[0] == y_target.shape[0]

# Compare sparse storage with dense storage.
dense_bytes = X_sparse.toarray().nbytes
sparse_bytes = X_sparse.data.nbytes + X_sparse.indices.nbytes + X_sparse.indptr.nbytes

# Show compact inspection results only.
print(f"X shape: {X_sparse.shape}, y shape: {y_target.shape}")
print(f"Two dimensional X: {is_two_dimensional}")
print(f"Matching sample count: {matching_samples}")
print(f"Nonzero entries: {X_sparse.nnz}")
print(f"Dense bytes: {dense_bytes}, sparse bytes: {sparse_bytes}")
print(f"Sparse format: {X_sparse.getformat()}")



## **2. Shapes and Dtypes**

### **2.1. Feature Matrix Shape**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_02_01.jpg?v=1783903131" width="250">



>* Rows are samples; columns are features
>* Single-feature data still needs two dimensions

>* Single-column selections can collapse feature matrices
>* Rows are samples; columns are features

>* Keep feature columns consistent across datasets
>* Verify two-dimensional sample-by-feature structure



In [ ]:
#@title Python Code - Feature Matrix Shape

# Feature matrices need two dimensions.
# Rows are samples, columns features.
# Check shapes before fitting estimators.

import numpy as np
import pandas as pd

# Create one feature measured for houses.
house_sizes_sqft = [850, 1200, 1500, 1800]

# A plain array becomes one dimensional.
wrong_vector = np.array(house_sizes_sqft)

# Reshape into rows and one column.
right_matrix = wrong_vector.reshape(-1, 1)

# Build a small two feature table.
houses = pd.DataFrame({
    "size_sqft": house_sizes_sqft,
    "distance_km": [2.1, 5.4, 3.2, 8.0],
})

# Selecting one column returns a Series.
series_choice = houses["size_sqft"]

# Double brackets preserve a DataFrame.
frame_choice = houses[["size_sqft"]]

# Summarize each container shape safely.
print("wrong_vector shape:", wrong_vector.shape)
print("right_matrix shape:", right_matrix.shape)
print("full DataFrame shape:", houses.shape)
print("single Series shape:", series_choice.shape)
print("single DataFrame shape:", frame_choice.shape)

# Check the feature matrix contract.
checks = [
    ("wrong_vector", wrong_vector.ndim == 2),
    ("right_matrix", right_matrix.ndim == 2),
    ("frame_choice", frame_choice.ndim == 2),
]

# Print compact pass or fail results.
for name, is_matrix in checks:
    status = "OK" if is_matrix else "Needs reshape"
    print(name, "->", status)

# Confirm rows match the target length.
target_prices = np.array([210000, 280000, 330000, 390000])
print("rows match target:", right_matrix.shape[0] == target_prices.shape[0])



### **2.2. Target Shape Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_02_02.jpg?v=1783903134" width="250">



>* Match one target value per sample
>* Avoid accidental column-shaped targets

>* Match target shape to prediction task
>* Separate accidental dimensions from multi-output targets

>* Data prep can distort target shapes
>* Verify alignment, dimensions, and multi-column intent



In [ ]:
#@title Python Code - Target Shape Checks

# Target shapes protect supervised learning workflows.
# Each feature row needs matching labels.
# Extra dimensions can change task meaning.

import numpy as np
import pandas as pd

# Create tiny feature data with four samples.
features = pd.DataFrame(
    {"height_cm": [170, 165, 180, 175],
     "weight_lb": [150, 130, 180, 160]}
)

# Select the target as a Series.
y_series = features["height_cm"] > 172

# Select the target as a one-column DataFrame.
y_column = features[["height_cm"]] > 172

# Create an intentionally multi-output target.
y_multi = pd.DataFrame(
    {"tall": y_series,
     "heavy": features["weight_lb"] > 155}
)

# Define a compact target inspection helper.
def describe_target(name, target, expected_rows):
    arr = np.asarray(target)
    rows_match = arr.shape[0] == expected_rows
    one_dimensional = arr.ndim == 1
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}")
    print(f"  rows match X: {rows_match}, ordinary target: {one_dimensional}")

# Check the feature row count first.
expected_rows = len(features)

# Inspect common target extraction outcomes.
describe_target("Series target", y_series, expected_rows)
describe_target("Column DataFrame", y_column, expected_rows)
describe_target("Multi-output target", y_multi, expected_rows)

# Show the safest ordinary target conversion.
y_fixed = np.asarray(y_column).ravel()
print(f"Fixed ordinary target shape: {y_fixed.shape}")



### **2.3. Dtype Checks**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_02_03.jpg?v=1783903129" width="250">



>* Check feature dtypes before fitting estimators
>* Convert misleading text or objects to numbers

>* Transform nonnumeric features before modeling
>* Check encodings preserve real meaning

>* Target dtype defines the learning task
>* Check consistency, formatting, and missing labels



In [ ]:
#@title Python Code - Dtype Checks

# Dtype checks prevent confusing estimator errors.
# Small tables make hidden problems visible.
# Targets need consistent shapes and dtypes.

import numpy as np
import pandas as pd

# Create features with intentional dtype problems.
features = pd.DataFrame({
    "area_sqft": [850, 920, "1,050", 1100],
    "price_usd": [250000, "$275000", 310000, 330000],
    "has_garage": [True, False, "yes", True],
})

# Create a target with mixed label types.
target = pd.Series(["sold", "not_sold", 1, "sold"])

# Inspect compact dtype information before fitting.
print("Feature dtypes:", features.dtypes.astype(str).to_dict())

# Detect columns stored as generic objects.
object_columns = features.select_dtypes(include="object").columns.tolist()
print("Object feature columns:", object_columns)

# Convert numeric-looking columns safely and explicitly.
clean_area = features["area_sqft"].astype(str).str.replace(",", "")
clean_price = features["price_usd"].astype(str).str.replace("$", "", regex=False)

# Build a numeric feature matrix for estimators.
X_numeric = pd.DataFrame({
    "area_sqft": pd.to_numeric(clean_area, errors="coerce"),
    "price_usd": pd.to_numeric(clean_price, errors="coerce"),
    "has_garage": features["has_garage"].map({True: 1, False: 0, "yes": 1}),
})

# Check missing values created during conversion.
print("Missing values after conversion:", X_numeric.isna().sum().to_dict())

# Confirm estimator-friendly shape and dtype summary.
print("X shape and dtypes:", X_numeric.shape, X_numeric.dtypes.astype(str).to_dict())

# Inspect target shape and mixed Python types.
y_array = target.to_numpy()
target_types = sorted({type(value).__name__ for value in y_array})
print("Target shape and types:", y_array.shape, target_types)

# Flag inconsistent target labels before fitting.
consistent_target = len(target_types) == 1
print("Target labels consistent:", consistent_target)

# Show a safe numeric array preview only.
print("Numeric array shape:", X_numeric.to_numpy(dtype=float).shape)



## **3. Inspection Utilities**

### **3.1. Bunch Data Containers**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_03_01.jpg?v=1783903137" width="250">



>* Bunch stores dataset parts together
>* Inspect meaning and shape before modeling

>* Reveal built-in dataset structure clearly
>* Separate features, labels, and task context

>* Metadata makes model behavior easier to explain
>* Bunches keep datasets structured and inspectable



In [ ]:
#@title Python Code - Bunch Data Containers

# Bunch containers keep dataset pieces together.
# Attribute access makes inspection beginner friendly.
# Metadata helps explain arrays before modeling.

from sklearn.datasets import load_iris
from sklearn.utils import Bunch

# Load a tiny built-in Bunch dataset.
iris_bunch = load_iris()

# Inspect dictionary-style and attribute-style access.
container_type = type(iris_bunch).__name__
key_names = list(iris_bunch.keys())

# Check shapes before any estimator receives data.
feature_shape = iris_bunch.data.shape
target_shape = iris_bunch.target.shape

# Read human-friendly metadata from the container.
first_features = list(iris_bunch.feature_names[:2])
class_names = list(iris_bunch.target_names)

# Build a tiny custom Bunch for comparison.
housing_bunch = Bunch(
    data=[[850, 2], [1200, 3], [1600, 4]],
    target=[250000, 340000, 430000],
    feature_names=["square_feet", "bedrooms"],
    DESCR="Tiny housing example using Imperial units.",
)

# Validate the custom container has aligned rows.
row_count = len(housing_bunch.data)
target_count = len(housing_bunch.target)
shape_ok = row_count == target_count

# Print compact inspection results only.
print(f"Container type: {container_type}")
print(f"Available keys: {key_names[:5]}")
print(f"Iris data shape: {feature_shape}")
print(f"Iris target shape: {target_shape}")
print(f"First feature names: {first_features}")
print(f"Target names: {class_names}")
print(f"Custom feature names: {housing_bunch.feature_names}")
print(f"Custom rows match targets: {shape_ok}")



### **3.2. Cloning Estimators**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_03_02.jpg?v=1783903139" width="250">



>* Copies estimator settings, not learned results
>* Creates a clean model for refitting

>* Clean clones prevent cross-validation data leakage
>* Same settings enable fair repeated comparisons

>* Separate setup parameters from learned attributes
>* Reveal estimator design issues for inspection



### **3.3. Feature Name Inspection**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_02/Lecture_B/image_03_03.jpg?v=1783903141" width="250">



>* Check column names, not just shapes
>* Catch reordered features before silent errors

>* Track names through preprocessing transformations
>* Link model inputs to real-world meaning

>* Compare feature names to catch data drift
>* Use names to audit model inputs



# <font color="#418FDE" size="6.5" uppercase>**Data Contracts**</font>


In this lecture, you learned to:
- Prepare NumPy, pandas, and sparse inputs that satisfy scikit-learn shape requirements. 
- Identify dtype and target-shape issues before fitting estimators. 
- Use dataset containers, feature names, cloning, tags, and diagrams for inspection. 

In the next Module (Module 3), we will go over 'None'